In [ ]:
!pip install langchain==0.3.25 langchain-groq langchain-community langchain-text-splitters langchain-huggingface chromadb pypdf

In [ ]:
!pip install wandb -q
print(" W&B installed!")

In [ ]:
import wandb
import os
import time

# Login to W&B
wandb.login(key="your-wandb-key")

# Start a monitoring run
run = wandb.init(
    project="RAG-monitoring",
    name="rag-app-session"
)

print(" W&B monitoring activated!")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]

loader = PyPDFLoader(filename)
documents = loader.load()
print(f" Pages loaded: {len(documents)}")

In [ ]:
def monitored_qa(question):
    start_time = time.time()

    # Get answer
    result = qa_chain.invoke({"query": question})

    end_time = time.time()
    response_time = end_time - start_time

    answer = result["result"]

    # Log to W&B dashboard
    wandb.log({
        "question": question,
        "answer": answer,
        "response_time_seconds": response_time,
        "num_source_chunks": len(result["source_documents"]),
        "answer_length": len(answer)
    })

    return answer, response_time

# Test with questions
questions = [
    "What is this document about?",
    "Who is the author?",
    "What methodology was used?"
]

for q in questions:
    answer, time_taken = monitored_qa(q)
    print(f"\n {q}")
    print(f" {answer[:200]}...")
    print(f" Response time: {time_taken:.2f} seconds")
    print("-" * 50)

print("\n All queries logged to W&B dashboard!")